In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import json
import pathlib
import dotenv
import pickle
import sys
import json5
import csv
import os
import pandas as pd
import numpy as np
from typing import Dict, List
from ast import literal_eval
from datetime import datetime
from pydantic import BaseModel, Field, ValidationError, parse_obj_as

current_dir = pathlib.Path().parent.resolve()
sys.path.append(str(current_dir.parent))

from loguru import logger
# from datahub_integrations.gen_ai.bedrock import BedrockModel
from datahub_integrations.gen_ai.term_suggestion_v2 import get_term_recommendations
from datahub_integrations.gen_ai.description_v2 import (
    extract_metadata_for_urn,
    transform_table_info_for_llm,
)
from datahub_integrations.gen_ai.term_suggestion_v2_context import (
    fetch_glossary_info,
    GlossaryUniverseConfig,
)

from docs_generation.graph_helper import create_datahub_graph
from helper import write_llm_output_to_csv

from term_suggestion_analysis_helper import (
    get_table_and_column_infos_dict, 
    get_failed_response_table_urns, 
    read_labeled_column_data,
    get_prediction_df,
    filter_predictions_df,
    get_classification_report_df
)
    
# TERM_SUGGESTION_GENERATION_MODEL: BedrockModel = parse_obj_as(
#     BedrockModel,
#     os.getenv(
#         "TERM_SUGGESTION_GENERATION_BEDROCK_MODEL", BedrockModel.CLAUDE_3_HAIKU.value
#     ),
# )

dotenv.load_dotenv(current_dir / ".env")

True

In [164]:
QUERY = """query getSearchResultsForMultiple($input: SearchAcrossEntitiesInput!) {
  searchAcrossEntities(input: $input) {
    start
    count
    total
    searchResults {
      entity {
        urn
        ... on Dataset {
          properties {
            description
          }
          editableProperties {
            description
          }
          schemaMetadata {
            fields {
              fieldPath
              description
              glossaryTerms {
                terms {
                  term {
                    urn
                  }
                }
              }
            }
          }
          editableSchemaMetadata {
            editableSchemaFieldInfo {
              fieldPath
              description
              glossaryTerms {
                terms {
                  term {
                    urn
                  }
                }
              }
            }
          }
        }
        __typename
      }
      matchedFields {
        name
        value
        __typename
      }
      __typename
    }
    __typename
  }
}"""

In [165]:
BASE_DIRECTORY = current_dir / "care"
# COLUMN_LABELS_CSV_PATH = current_dir / "column_labels_with_care_data.csv"
# URNS_DICT_PATH = current_dir / "test_urns.json"

# INSTANCES_TO_EXAMINE = ['longtailcompanions', 'acryl', 'chime', 'notion', 'twilio']


# EXPERIMENT_NAME = "term_suggestion_output"
GLOSSARY_INSTANCE = "care"

# URNS_DICT: Dict[str, List[str]] = json5.loads(  # type: ignore
#     (URNS_DICT_PATH).read_text()
# )
    
# URNS_DICT = {key:value for key, value in URNS_DICT.items() if key in INSTANCES_TO_EXAMINE}

In [7]:
# table_infos_dict, column_infos_dict = get_table_and_column_infos_dict(urns_dict=URNS_DICT)

urn:li:dataset:(urn:li:dataPlatform:snowflake,datahub_community.orbit_fivetran.member_in_workspace,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,datahub_community.orbit.workspace_activity,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,analytics.looker.zendesk_agent,PROD)
urn:li:dataset:(urn:li:dataPlatform:looker,snowflake.view.penny_claim_created,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,analytics.omx.disputes_omxri,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,analytics.looker.user_socure_cip_dim,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,analytics.omx.five9_call_acd_sla,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,analytics.omx.five9_call_acd_sla_view,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,postgres_db.mms.external_card_transfers,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,edw_db.core.fct_settled_transaction,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,production.core.stripe_customer_daily,PROD)
urn:li:dataset:(ur

Ignoring unknown aspect type entityInferenceMetadata
Ignoring unknown aspect type entityInferenceMetadata


urn:li:dataset:(urn:li:dataPlatform:snowflake,long_tail_companions.adoption.human_profiles,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,long_tail_companions.adoption.pets,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,long_tail_companions.analytics.active_customer_ltv,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,long_tail_companions.analytics.dog_breeds,PROD)
urn:li:dataset:(urn:li:dataPlatform:looker,analytics.explore.retention_cohort,PROD)
urn:li:dataset:(urn:li:dataPlatform:looker,long_tail_companions.view.pet_details,PROD)
urn:li:dataset:(urn:li:dataPlatform:looker,long_tail_companions.explore.monthly_adoption_projection,PROD)
urn:li:dataset:(urn:li:dataPlatform:redshift,banking.public.account_transactions,PROD)


In [ ]:
# prompt = (current_dir / "../../src/datahub_integrations/gen_ai/term_suggestion_prompt.txt").read_text()
# prompt

# Fetch Glossary terms

In [4]:
GLOSSARY_INSTANCE = "care"
GLOSSARY_NODES = ["urn:li:glossaryNode:d966ad51-49a7-411c-8d06-2ee2dd210647", "urn:li:glossaryNode:09effdec-8810-415c-ace7-4e53090c502c"]
GLOSSARY_TERMS = []

graph_client = create_datahub_graph(GLOSSARY_INSTANCE)
glossary_config = GlossaryUniverseConfig(
    glossary_nodes=GLOSSARY_NODES,
    glossary_terms=GLOSSARY_TERMS
)
glossary_info = fetch_glossary_info(graph_client=graph_client, universe=glossary_config)

In [169]:
# glossary_info

In [180]:
from tqdm import tqdm
glossary_query_results = {}
for term_urn in tqdm(glossary_info.glossary.keys(), total = len(glossary_info.glossary.keys())):
    variables = {
      "input": {
        "types": [],
        "query": "*",
        "start": 0,
        "count": 5,
        "orFilters": [
          {
            "and": [
              {
                "field": "fieldGlossaryTerms",
                "values": [
                  term_urn
                ]
              }
            ]
          }
        ],
        "searchFlags": {
          "skipCache": True
        }
      }
    }
    graph_client = create_datahub_graph("care")
    glossary_query_results[term_urn] = {"term_name": glossary_info.glossary[term_urn]["term_name"], "query_value": graph_client.execute_graphql(query=QUERY, variables=variables)}

100%|██████████████████████████████████████████████████████████████████████████████████| 47/47 [00:18<00:00,  2.54it/s]


In [171]:
glossary_query_results.keys()

dict_keys(['urn:li:glossaryTerm:allergies', 'urn:li:glossaryTerm:password_encrypted', 'urn:li:glossaryTerm:iban', 'urn:li:glossaryTerm:credit_card_token', 'urn:li:glossaryTerm:password_cleartext', 'urn:li:glossaryTerm:id_number_drivers_license', 'urn:li:glossaryTerm:id_number_gov', 'urn:li:glossaryTerm:id_number_tax', 'urn:li:glossaryTerm:id_number_ssn', 'urn:li:glossaryTerm:id_user', 'urn:li:glossaryTerm:health_info', 'urn:li:glossaryTerm:credit_card_number', 'urn:li:glossaryTerm:bank_routing_number', 'urn:li:glossaryTerm:bank_account_number', 'urn:li:glossaryTerm:authentication_token', 'urn:li:glossaryTerm:username', 'urn:li:glossaryTerm:name_first', 'urn:li:glossaryTerm:name_middle', 'urn:li:glossaryTerm:name_fullname', 'urn:li:glossaryTerm:person_name', 'urn:li:glossaryTerm:name_last', 'urn:li:glossaryTerm:address_ip', 'urn:li:glossaryTerm:member_messages', 'urn:li:glossaryTerm:geo_location', 'urn:li:glossaryTerm:id_socialmedia', 'urn:li:glossaryTerm:gender', 'urn:li:glossaryTerm:p

In [172]:
glossary_query_results[list(glossary_query_results.keys())[0]]

{'term_name': 'Allergies',
 'query_value': {'searchAcrossEntities': {'start': 0,
   'count': 10,
   'total': 35,
   'searchResults': [{'entity': {'urn': 'urn:li:dataset:(urn:li:dataPlatform:mysql,backup_care.MEMBER_SEGMENTATION_CHILDREN_DETAILS,PROD)',
      'properties': {'description': 'Member Segmentation Children Details'},
      'editableProperties': None,
      'schemaMetadata': {'fields': [{'fieldPath': 'ID',
         'description': 'Identifier to uniquely identify the Member Segmentation Children Details',
         'glossaryTerms': None},
        {'fieldPath': 'MEMBER_SEGMENTATION_DETAILS_ID',
         'description': 'Reference to the parent Member Segmentation Details',
         'glossaryTerms': None},
        {'fieldPath': 'FIRST_NAME',
         'description': 'first name of the child',
         'glossaryTerms': {'terms': [{'term': {'urn': 'urn:li:glossaryTerm:name_first'}}]}},
        {'fieldPath': 'DOB_MONTH',
         'description': 'Month of Birth',
         'glossaryTerm

In [181]:
_COLUMN_INFO_FOR_PROMPT = [
    "column_name",
    "descriptions",
    "sample_values",
    "datatype",
]
glossary_example_columns = {}
for key, value in tqdm(glossary_query_results.items(), total = len(glossary_query_results)):
    found_column = False
    print(key)
    glossary_term_name = value["term_name"]
    for result in value["query_value"].get("searchAcrossEntities", {}).get("searchResults", {}):
        try:
            entity = result.get("entity", {})
            table_urn = entity["urn"]
    #         print(table_urn)
            table_entity = graph_client.get_entity_semityped(table_urn)
            extracted_table_info = extract_metadata_for_urn(table_entity, table_urn, graph_client)
            table_info, column_info = transform_table_info_for_llm(extracted_table_info)
    #         print(column_info)
            for column in column_info.keys():
                column_info[column].update(
                    {
                        "datatype": column_info[column]
                        .get("metadata", {})
                        .get("nativeDataType", "")
                    }
                )
            column_info_filtered = {}
            for column_name, column_info_dict in column_info.items():
                column_info_filtered[column_name] = {
                    k: v for k, v in column_info_dict.items() if k in _COLUMN_INFO_FOR_PROMPT
                }
            for column in column_info_filtered.keys():
                column_glossary_terms = column_info[column].get("glossary_terms")
                if isinstance(column_glossary_terms, list):
                    for term in column_glossary_terms:
                        if term["term_name"] == glossary_term_name:
                            print(term["term_name"])
                            if key in glossary_example_columns.keys():
                                glossary_example_columns[key].append(column_info_filtered[column])
                            else:
                                glossary_example_columns[key] = [column_info_filtered[column]]
                            found_column = True
#                             break
                        else:
                            continue
#                 if found_column == True:
#                     break
#             if found_column == True:
#                 break
        except Exception as e:
            print(f"Exception occurred!! {e}")
            continue

  0%|                                                                                           | 0/47 [00:00<?, ?it/s]

urn:li:glossaryTerm:allergies
Allergies
Allergies
Allergies
Allergies
Allergies
Allergies
Allergies
Allergies
Allergies


  2%|█▊                                                                                 | 1/47 [00:03<02:34,  3.36s/it]

Allergies
Allergies
urn:li:glossaryTerm:password_encrypted
Password - Encrypted
Password - Encrypted
Password - Encrypted
Password - Encrypted
Password - Encrypted
Password - Encrypted
Password - Encrypted
Password - Encrypted


  4%|███▌                                                                               | 2/47 [00:06<02:24,  3.21s/it]

...

Member ID
urn:li:glossaryTerm:birth_day
urn:li:glossaryTerm:birth_year
urn:li:glossaryTerm:birth_fulldate
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date


 96%|██████████████████████████████████████████████████████████████████████████████▌   | 45/47 [01:51<00:04,  2.48s/it]

Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
Birth Date
urn:li:glossaryTerm:birth_month
urn:li:glossaryTerm:id_memberuuid
Member UUID
Member UUID
Member UUID
Member UUID


100%|██████████████████████████████████████████████████████████████████████████████████| 47/47 [01:55<00:00,  2.45s/it]

Member UUID


In [176]:
# for key, value in glossary_example_columns.items():
#     if value.get("description") is None:
#         glossary_example_columns[key]["description"] = ""
#     if value.get("sample_values") is None:
#         glossary_term_example_columns[key]["sample_values"] = []

In [179]:
glossary_info.glossary

{'urn:li:glossaryTerm:allergies': {'term_name': 'Allergies',
  'term_description': '',
  'parent_node': {'parent_name': 'PII - Level 1',
   'parent_description': 'PII classified by [Care.com](//Care.com) as Level 1 which includes:\n\nAny Government ID number including Social Security and Drivers License, Passport\n\nName in combination with Date of Birth\n\nMedical Information (ePHI)\n\nCredentials - username / password'}},
 'urn:li:glossaryTerm:password_encrypted': {'term_name': 'Password - Encrypted',
  'term_description': '',
  'parent_node': {'parent_name': 'PII - Level 1',
   'parent_description': 'PII classified by [Care.com](//Care.com) as Level 1 which includes:\n\nAny Government ID number including Social Security and Drivers License, Passport\n\nName in combination with Date of Birth\n\nMedical Information (ePHI)\n\nCredentials - username / password'}},
 'urn:li:glossaryTerm:iban': {'term_name': 'IBAN',
  'term_description': '',
  'parent_node': {'parent_name': 'PII - Level 1

In [182]:
glossary_example_columns

{'urn:li:glossaryTerm:allergies': [{'column_name': 'E_ALLERGY_NOTE',
   'descriptions': 'encrypted data of ALLERGY_NOTE',
   'datatype': 'BLOB'},
  {'column_name': 'Allergies', 'datatype': 'VARCHAR(500)'},
  {'column_name': 'CatAllergies', 'datatype': 'VARCHAR(10)'},
  {'column_name': 'DogAllergies', 'datatype': 'VARCHAR(10)'},
  {'column_name': 'Allergies', 'datatype': 'VARCHAR(500)'},
  {'column_name': 'CatAllergies', 'datatype': 'INTEGER(1)'},
  {'column_name': 'DogAllergies', 'datatype': 'INTEGER(1)'},
  {'column_name': 'ALLERGY_TYPE',
   'descriptions': 'allergy type',
   'sample_values': ['MEDICATIONS',
    'FOOD',
    'RESPIRATORY',
    'BEE_STING',
    'OTHER',
    'MEDICATIONS',
    'FOOD',
    'RESPIRATORY',
    'BEE_STING',
    'OTHER',
    'MEDICATIONS',
    'FOOD',
    'RESPIRATORY',
    'BEE_STING',
    'OTHER',
    'MEDICATIONS',
    'FOOD',
    'RESPIRATORY',
    'BEE_STING',
    'OTHER'],
   'datatype': "ENUM('MEDICATIONS','FOOD','RESPIRATORY','BEE_STING','OTHER')"},
 

In [16]:
glossary_examples = {'urn:li:glossaryTerm:allergies': {'column_name': 'E_ALLERGY_NOTE',
   'descriptions': 'encrypted data of ALLERGY_NOTE',
   'datatype': 'BLOB'},
 'urn:li:glossaryTerm:password_encrypted': {'column_name': 'Password',
   'datatype': 'NVARCHAR(50) COLLATE SQL_Latin1_General_CP1_CI_AS'},
 'urn:li:glossaryTerm:id_number_drivers_license': {'column_name': 'DriversLicenseNumber',
   'datatype': 'NVARCHAR(25) COLLATE SQL_Latin1_General_CP1_CI_AS'},
 'urn:li:glossaryTerm:id_number_tax': {'column_name': 'TaxCollectionTypeID',
   'datatype': 'INTEGER'},
 'urn:li:glossaryTerm:id_number_ssn': {'column_name': 'SocialSecurityNumber',
   'datatype': 'NVARCHAR(512) COLLATE SQL_Latin1_General_CP1_CI_AS'},
 'urn:li:glossaryTerm:credit_card_number': {'column_name': 'Credit Card Number',
   'datatype': 'integer'},
 'urn:li:glossaryTerm:username': {'column_name': 'Username', 'datatype': 'STR'},
 'urn:li:glossaryTerm:name_first': {'column_name': 'First Name',
   'datatype': 'WSTR'},
 'urn:li:glossaryTerm:name_middle': {'column_name': 'MIDDLE_NAME_PROVIDED',
   'datatype': 'VARCHAR(10)'},
 'urn:li:glossaryTerm:name_fullname': {'column_name': 'Full Name',
   'datatype': 'STR'},
 'urn:li:glossaryTerm:name_last': {'column_name': 'Last Name',
   'datatype': 'WSTR'},
 'urn:li:glossaryTerm:address_ip': {'column_name': 'IP_ADDRESS', 'datatype': 'VARCHAR(15)'},
 'urn:li:glossaryTerm:geo_location': {'column_name': 'Latitude',
   'datatype': 'R8'},
 'urn:li:glossaryTerm:id_socialmedia': {'column_name': 'Apple Silicon',
   'datatype': 'STR'},
 'urn:li:glossaryTerm:gender': {'column_name': 'Sex', 'datatype': 'WSTR'},
 'urn:li:glossaryTerm:photo_personal': {'column_name': '[version=2.0].[type=properties].images',
   'datatype': 'properties'},
 'urn:li:glossaryTerm:id_number_employee': {'column_name': 'Employee ID',
   'datatype': 'WSTR'},
 'urn:li:glossaryTerm:phone_number': {'column_name': 'primary_phone', 'datatype': 'R8'},
 'urn:li:glossaryTerm:address_email': {'column_name': 'email_address', 'datatype': 'STR'},
 'urn:li:glossaryTerm:address_county': {'column_name': 'County',
   'datatype': 'NVARCHAR(30) COLLATE SQL_Latin1_General_CP1_CI_AS'},
 'urn:li:glossaryTerm:address_city': {'column_name': '[version=2.0].[type=text].city',
   'datatype': 'text'},
 'urn:li:glossaryTerm:address_state': {'column_name': '[version=2.0].[type=properties].map.[type=text].uiState',
   'datatype': 'text'},
 'urn:li:glossaryTerm:address_zipcode': {'column_name': '[version=2.0].[type=keyword].zip', 'datatype': 'keyword'},
 'urn:li:glossaryTerm:address_street': {'column_name': '[version=2.0].[type=text].addressLine1',
   'datatype': 'text'},
 'urn:li:glossaryTerm:address_country': {'column_name': 'COUNTRY',
   'datatype': 'WSTR'},
 'urn:li:glossaryTerm:id_member': {'column_name': 'MEMBER_ID', 'datatype': 'I8'},
 'urn:li:glossaryTerm:birth_fulldate': {'column_name': 'Date of Birth',
   'datatype': 'TIMESTAMPZ'},
 'urn:li:glossaryTerm:id_memberuuid': {'column_name': '[version=2.0].[type=keyword].providerUUID',
   'datatype': 'keyword'}}

In [17]:
for key, value in glossary_info.glossary.items():
    glossary_info.glossary[key]["example"] = glossary_examples.get(key, {})

In [18]:
term_descriptions = {}
for key, value in glossary_info.glossary.items():
    term_name = glossary_info.glossary[key]["term_name"]
    term_descriptions[term_name] = ""

In [19]:
term_descriptions["Allergies"] = "'Allergies' refers to medical information pertaining to an individual's adverse reactions to specific substances. This can include details about food allergies, medication sensitivities, and environmental allergens. In the context of data privacy, this information is classified as sensitive personal data (ePHI) and should be handled with appropriate security measures to protect individual privacy."
term_descriptions["Password - Encrypted"] = "'Password - Encrypted' refers to a user's password that has been securely transformed using encryption algorithms to protect it from unauthorized access. In a database context, this term is vital for fields where user authentication and data privacy are paramount. This term is essential for identifying columns that store sensitive user information, particularly in systems requiring secure user access, such as online services and applications. \n Some examples are pwd, user_pass, encrypted_pass, password_digest, pass_hash, credentials, login_password, user_secret, auth_token, password_value etc"
term_descriptions["IBAN"] = "'IBAN' (International Bank Account Number) refers to a standardized format for identifying bank accounts across countries. It is used internationally to facilitate cross-border transactions and ensure accurate and secure processing of payments. An IBAN consists of a country code, check digits, and a bank account number, making it essential for identifying a specific bank account in international financial operations. As sensitive personal information, the IBAN is classified under PII and should be handled with care to protect against unauthorized access. \n Some examples are: iban, account_iban, user_iban, customer_iban, bank_account_number, iban_code, iban_value, international_account_number etc"
term_descriptions["Credit Card Token"] = "'Credit Card Token' refers to a secure representation of a credit card number that is generated to replace the actual card number during transactions. This tokenization process enhances security by ensuring that sensitive credit card information is not stored or transmitted, thereby reducing the risk of fraud and data breaches. Common column names for storing this information include examples such as credit_card_token, tokenized_card_number, card_token, payment_token, secure_card_token, and transaction_token."
term_descriptions["Password - Cleartext"] = "'Password - Cleartext' refers to a user's password that is stored or transmitted in an unencrypted format, making it easily readable by anyone with access to the data. This practice poses significant security risks, as cleartext passwords can be intercepted or accessed, leading to unauthorized account access and potential data breaches. It is crucial to avoid storing passwords in cleartext to protect sensitive personal information. Common column names for representing this type of data include examples such as cleartext_password, raw_password, user_password_plain, password_value, and account_password."
term_descriptions["Driving License Number"] = "'Driving License Number' refers to the unique identifier assigned to an individual's driving license by a governmental authority. This number is used to verify the identity of the license holder and is essential for legal driving and identification purposes. It is classified as sensitive personal information and should be handled with care to protect against unauthorized access. Common column names for storing this information include examples such as drivers_license_number, dl_number, license_id, driver_license, and license_number."
term_descriptions["Government ID"] = "'Government ID' refers to any identification document issued by a governmental authority to verify an individual's identity. This can include various types of IDs such as passports, national identity cards, and driver's licenses. Government IDs are essential for various legal and administrative processes and are considered sensitive personal information that must be protected against unauthorized access. Common column names for storing this information include examples such as government_id, id_number, government_identification, national_id, and issued_id."
term_descriptions["Tax ID"] = "'Tax ID' refers to a unique identification number assigned to individuals or businesses by tax authorities for tax purposes. This number is used to track tax obligations and compliance and is essential for filing tax returns and conducting financial transactions. Tax IDs are considered sensitive personal information and should be safeguarded to prevent unauthorized access. Common column names for storing this information include examples such as tax_id, tax_identification_number, taxpayer_id, income_tax_id, and tax_collection_id."
term_descriptions["Social Security Number"] = "'Social Security Number' refers to a unique identifier assigned to individuals by the government for tracking earnings, benefits, and taxation. It is used primarily in the United States and is essential for various legal and financial processes, including tax reporting and eligibility for government services. Given its sensitivity, the Social Security Number must be handled with strict confidentiality to prevent identity theft. Common column names for storing this information include examples such as social_security_number, ssn, taxpayer_ssn, social_sec_num, and ssn_value."
term_descriptions["User ID"] = "'User ID' refers to a unique identifier assigned to an individual user within a system or platform. It is typically used to track user activities, preferences, and account information. This identifier plays a crucial role in user authentication and access control. As it is considered sensitive personal information, it should be protected against unauthorized access. Common column names for storing this information include examples such as user_id, account_id, user_identifier, uid, and member_id."
term_descriptions["Health Information"] = "'Health Information' refers to any data related to an individual's physical or mental health status, healthcare services provided, or payment for healthcare. This information is critical for managing patient care and treatment and is subject to strict privacy regulations to protect individuals' rights. Given its sensitive nature, it is classified as personal identifiable information (PII) and must be securely stored and transmitted. Common column names for storing this information include examples such as health_info, medical_records, patient_health_data, health_history, and health_status."
term_descriptions["Credit / Debit Card Number"] = "'Credit / Debit Card Number' refers to the unique number assigned to a credit or debit card used for financial transactions. This number is essential for processing payments and is classified as sensitive personal information due to its potential for misuse. Protecting this information is crucial to prevent fraud and identity theft. Common column names for storing this information include examples such as credit_card_number, debit_card_number, card_number, payment_card_number, and financial_card_id."
term_descriptions["Bank Routing Number"] = "'Bank Routing Number' refers to a unique nine-digit code assigned to a financial institution in the United States, used to identify the institution in transactions such as wire transfers and direct deposits. This number is essential for processing payments and ensuring that funds are routed accurately. As it is considered sensitive personal information, it must be protected against unauthorized access to prevent fraud. Common column names for storing this information include examples such as bank_routing_number, routing_number, aba_number, financial_institution_code, and transit_number."
term_descriptions["Bank Account Number"] = "'Bank Account Number' refers to a unique identifier assigned to an individual or business bank account by a financial institution. This number is crucial for conducting transactions, such as deposits and withdrawals, and is often required for direct deposits and electronic payments. As it is considered sensitive personal information, it should be protected to prevent unauthorized access and fraud. Common column names for storing this information include examples such as bank_account_number, account_number, checking_account_number, savings_account_number, and financial_account_id."
term_descriptions["Authentication Token"] = "'Authentication Token' refers to a unique string of characters generated to verify a user's identity and authorize access to a system or service. These tokens are often used in authentication processes, replacing traditional username and password combinations to enhance security. Since they can grant access to sensitive information, authentication tokens are classified as personal identifiable information (PII) and must be securely stored and transmitted. Common column names for storing this information include examples such as authentication_token, access_token, user_token, session_token, and api_token."
term_descriptions["Username"] = "'Username' refers to a unique identifier chosen by a user to access a system or platform. It is often used in combination with a password to authenticate a user’s identity and is critical for account management and security. As it is considered sensitive personal information, it should be protected to prevent unauthorized access to user accounts. Common column names for storing this information include examples such as username, user_id, account_name, login_name, and user_handle."
term_descriptions["First Name"] = "'First Name' refers to the given name of an individual, used to identify and address them personally. It is a fundamental component of a person's identity and is often required in various forms of documentation and registration processes. As it is considered personal information, it should be handled with care to protect individual privacy. Common column names for storing this information include examples such as first_name, given_name, fname, user_first_name, and name_first."
term_descriptions["Middle Name"] = "'Middle Name' refers to any additional name(s) given to an individual, situated between their first and last names. While not always used, a middle name can carry cultural significance or honor family heritage. It is often included in official documents and forms to provide a fuller identification of the individual. Common column names for storing this information include examples such as middle_name, middle_initial, middle_name_provided, user_middle_name, and name_middle."
term_descriptions["Full Name"] = "'Full Name' refers to the complete name of an individual, typically consisting of their first name, middle name (if applicable), and last name. It is used for formal identification and documentation purposes. The full name is often required in legal, academic, and personal contexts to ensure clarity in identification. Common column names for storing this information include examples such as full_name, complete_name, user_full_name, name_complete, and entire_name."
term_descriptions["Person Name"] = "'Person Name' refers to the collective identifier for an individual, which may include their first name, middle name (if applicable), and last name. This term encompasses all components that make up a person's full legal name and is essential for identification in various contexts, including legal documents, personal records, and formal communications. Common column names for storing this information include examples such as person_name, individual_name, name_full, user_name, and complete_person_name."
term_descriptions["Last Name"] = "'Last Name' refers to the surname or family name of an individual, used to identify their lineage and family association. It is a key component of a person's full name and is often required in official documents, applications, and forms. As it is considered personal information, it should be handled with care to protect individual privacy. Common column names for storing this information include examples such as last_name, surname, family_name, user_last_name, and name_last."
term_descriptions["IP Address"] = "'IP Address' refers to a unique numerical label assigned to each device connected to a computer network that uses the Internet Protocol for communication. It serves two main functions: identifying the host or network interface and providing the location of the device in the network. As it can reveal information about a user's location and is considered sensitive information, it should be protected to prevent unauthorized access. Common column names for storing this information include examples such as ip_address, user_ip, device_ip, client_ip, and network_ip."
term_descriptions["Member Messages"] = "'Member Messages' refers to any communication or messaging content exchanged between users within a platform or system. This information can include direct messages, comments, or notifications and is essential for user interaction and engagement. Given its potential to contain personal information, it is classified as sensitive data and should be safeguarded against unauthorized access. Common column names for storing this information include examples such as member_messages, user_messages, chat_content, direct_messages, and message_history."
term_descriptions["Geolocation"] = "'Geolocation' refers to the identification or estimation of the real-world geographic location of an object, often represented by latitude and longitude coordinates. This information is crucial for location-based services, mapping applications, and user tracking. Due to its sensitivity and potential to reveal personal whereabouts, geolocation data is classified as personal identifiable information (PII) and must be protected. Common column names for storing this information include examples such as geolocation, latitude, longitude, user_location, and geo_coordinates."
term_descriptions["Social Media ID"] = "'Social Media ID' refers to a unique identifier assigned to a user's account on social media platforms. This ID is essential for user authentication, interaction, and accessing personalized content within the platform. As it can be linked to personal profiles and activities, it is considered sensitive information and should be protected to ensure user privacy. Common column names for storing this information include examples such as social_media_id, user_social_id, account_id, profile_id, and social_account_identifier."
term_descriptions["Gender"] = "'Gender' refers to the classification of individuals based on their identity as male, female, or other gender identities. It is an important demographic characteristic that can inform various analyses and services. Given its personal nature, gender information should be handled with sensitivity to ensure privacy and respect for individual identity. Common column names for storing this information include examples such as gender, sex, user_gender, gender_identity, and gender_category."
term_descriptions["Personal Photo"] = "'Personal Photo' refers to an image or photograph that depicts an individual, often used for identification, profiles, or social interactions. This type of sensitive data can reveal a person's identity and should be managed with care to protect privacy. Given its personal nature, the storage and sharing of personal photos must comply with data protection regulations. Common column names for storing this information include examples such as personal_photo, profile_picture, user_image, avatar, and photo_id."
term_descriptions["Employee ID"] = "'Employee ID' refers to a unique identifier assigned to each employee within an organization. This ID is used for tracking employee records, payroll, and access to company resources. As it is considered personal information, it should be protected to ensure confidentiality and prevent unauthorized access. Common column names for storing this information include examples such as employee_id, staff_id, user_id, worker_id, and emp_identifier."
term_descriptions["Geo Location - Coarse"] = "'Geo Location - Coarse' refers to an approximate geographic location identified using broader geographic indicators, such as city or region, rather than precise coordinates. This information can be useful for understanding general user demographics and behaviors but offers less specificity than detailed geolocation data. Given its potential to reveal personal information, it should be handled with care. Common column names for storing this information include examples such as geo_location_coarse, approximate_location, user_region, city_location, and region_identifier."
term_descriptions["Geo Location - Precise"] = "'Geo Location - Precise' refers to exact geographic coordinates, typically expressed in latitude and longitude. This data provides detailed location information that can be critical for applications requiring accurate positioning. Example column names include latitude, longitude, precise_location, and geo_coordinates."
term_descriptions["Phone Number"] = "'Phone Number' captures the telecommunication number assigned to an individual or entity, used for voice communication and contact purposes. Proper formatting is crucial for usability. Example column names include primary_phone, contact_number, mobile_number, and home_phone."
term_descriptions["Income"] = "'Income' represents the monetary earnings received by an individual over a specified period, typically used for assessing financial status or eligibility for services. Example column names include annual_income, monthly_income, income_amount, and salary."
term_descriptions["Email Address"] = "'Email Address' denotes the electronic mail identifier used to send and receive messages via the internet. It is essential for digital communication. Example column names include email_address, user_email, contact_email, and personal_email."
term_descriptions["County"] = "'County' specifies the administrative division within a state or region, often used in demographic and geographic data. Example column names include County, county_name, user_county, and residence_county."
term_descriptions["City"] = "'City' identifies the municipal entity within which a person resides, serving as a key geographic marker. Example column names include city, residence_city, user_city, and location_city."
term_descriptions["Address"] = "'Address' encompasses all components that define a specific location where an individual or entity resides or operates. Example column names include address_full, user_address, mailing_address, and location_address."
term_descriptions["State"] = "'State' indicates the regional division within a country, used to classify and sort geographic data. Example column names include state, state_name, user_state, and location_state."
term_descriptions["Zip Code"] = "'Zip Code' refers to the postal code used by postal services to identify specific geographic delivery areas. Example column names include zip_code, postal_code, user_zip, and location_zip."
term_descriptions["Street Address"] = "'Street Address' specifies the actual street location of a residence or business, forming part of the complete address. Example column names include street_address, address_line1, user_street, and location_street."
term_descriptions["Country"] = "'Country' designates the sovereign state in which an individual resides, crucial for geographic and demographic classifications. Example column names include country, country_name, user_country, and location_country."
term_descriptions["Marital Status"] = "'Marital Status' reflects the legal relationship status of an individual, which can influence various aspects of personal and legal matters. Example column names include marital_status, status, relationship_status, and user_marital_status."
term_descriptions["Member ID"] = "'Member ID' is a unique identifier assigned to individuals within a membership-based service, facilitating the tracking of member activities and privileges. Example column names include MEMBER_ID, member_identifier, user_id, and membership_number."
term_descriptions["Birth Day"] = "'Birth Day' refers to the specific day of an individual's birth, a crucial part of personal identification. Example column names include birth_day, day_of_birth, user_birth_day, and dob_day."
term_descriptions["Birth Year"] = "'Birth Year' indicates the year an individual was born, providing context for age-related data. Example column names include birth_year, year_of_birth, user_birth_year, and dob_year."
term_descriptions["Birth Date"] = "'Birth Date' specifies the complete date of birth, crucial for age verification and identity. Example column names include Date_of_Birth, dob, user_birth_date, and birthdate."
term_descriptions["Birth Month"] = "'Birth Month' indicates the month in which an individual was born, relevant for astrological and demographic analysis. Example column names include birth_month, month_of_birth, user_birth_month, and dob_month."
term_descriptions["Member UUID"] = "'Member UUID' is a universally unique identifier assigned to members within a system, ensuring distinct identification across databases. Example column names include providerUUID, member_uuid, uuid, and user_uuid."

In [20]:
for key, value in glossary_info.glossary.items():
    term_name = value["term_name"]
    if term_name in term_descriptions.keys():
        glossary_info.glossary[key]["term_description"] = term_descriptions[term_name]
    else:
        print(term_name)

In [21]:
with open("glossary_terms_with_term_descriptions_and_examples.pkl", 'wb') as f:
    pickle.dump(glossary_info, f)
    f.close()

In [23]:
for key, value in glossary_info.glossary.items():
    print(value["term_name"])
    print(value["term_description"], "\n\n")
    

Allergies
'Allergies' refers to medical information pertaining to an individual's adverse reactions to specific substances. This can include details about food allergies, medication sensitivities, and environmental allergens. In the context of data privacy, this information is classified as sensitive personal data (ePHI) and should be handled with appropriate security measures to protect individual privacy. 


Password - Encrypted
'Password - Encrypted' refers to a user's password that has been securely transformed using encryption algorithms to protect it from unauthorized access. In a database context, this term is vital for fields where user authentication and data privacy are paramount. This term is essential for identifying columns that store sensitive user information, particularly in systems requiring secure user access, such as online services and applications. 
 Some examples are pwd, user_pass, encrypted_pass, password_digest, pass_hash, credentials, login_password, user_secr

In [192]:

# 'urn:li:glossaryTerm:id_user'
# 'urn:li:glossaryTerm:id_member'
glossary_info.glossary['urn:li:glossaryTerm:allergies']["term_description"] = "'Allergies' refers to medical information pertaining to an individual's adverse reactions to specific substances. This can include details about food allergies, medication sensitivities, and environmental allergens. In the context of data privacy, this information is classified as sensitive personal data (ePHI) and should be handled with appropriate security measures to protect individual privacy."

glossary_info.glossary['urn:li:glossaryTerm:id_user']["term_description"] = "A unique identifier assigned to an individual user within a system or platform. It is typically used to track user activities, preferences, and account information."
glossary_info.glossary['urn:li:glossaryTerm:id_member']["term_description"] = "A distinctive identifier given to individuals enrolled in a membership-based service, allowing the organization to track member-related activities, privileges, and account status."

In [194]:
len(glossary_examples)

28

In [229]:
df = pd.read_csv("care_columns_ground_truth_without_comment.csv")
used_terms = df["glossary_terms_updated"].unique()
used_terms = set([y.strip() for x in used_terms if not isinstance(x, float) for y in x.split("/")])

terms_with_example = []
for key in glossary_examples.keys():
    terms_with_example.append(glossary_info.glossary[key]["term_name"])
used_terms_with_example = [term for term in used_terms if term in terms_with_example]
used_terms_without_example = [term for term in used_terms if term not in used_terms_with_example]
    
print("Total_used_terms\n", len(used_terms), "\n", used_terms)
print("\nTerms_with_example\n", len(terms_with_example), "\n", terms_with_example)
print("\nUsed_terms_with_example\n", len(used_terms_with_example), "\n", used_terms_with_example)
print("\nUsed_terms_without_example\n", len(used_terms_without_example), "\n", used_terms_without_example)

Total_used_terms
 16 
 {'User ID', 'Address', 'Personal Photo', 'Account Number', 'Member UUID', 'Last Name', 'Middle Name', 'Member ID', 'Zip Code', 'Gender', 'Username', 'Full Name', 'IP Address', 'Country', 'First Name', 'State'}

Terms_with_example
 28 
 ['Allergies', 'Password - Encrypted', 'Driving License Number', 'Tax ID', 'Social Security Number', 'Credit / Debit Card Number', 'Username', 'First Name', 'Middle Name', 'Full Name', 'Last Name', 'IP Address', 'Geolocation', 'Social Media ID', 'Gender', 'Personal Photo', 'Employee ID', 'Phone Number', 'Email Address', 'County', 'City', 'State', 'Zip Code', 'Street Address', 'Country', 'Member ID', 'Birth Date', 'Member UUID']

Used_terms_with_example
 13 
 ['Personal Photo', 'Member UUID', 'Last Name', 'Middle Name', 'Member ID', 'Zip Code', 'Gender', 'Username', 'Full Name', 'IP Address', 'Country', 'First Name', 'State']

Used_terms_without_example
 3 
 ['User ID', 'Address', 'Account Number']


In [ ]:
glossary_info

In [158]:
# glossary_examples = {}
# for key, value in glossary_query_results.items():
#     for result in value.get("searchAcrossEntities", {}).get("searchResults", {}):
#         entity = result.get("entity", {})
#         table_urn = entity["urn"]
#         for field in entity["schemaMetadata"]["fields"]:
#             glossary_terms_dict = field.get("glossaryTerms", {})
#             if isinstance(glossary_terms_dict, dict):
#                 glossary_terms = glossary_terms_dict["terms"]
#                 for terms in glossary_terms:
#                     term_urn = terms["term"]["urn"]
#                     if key in glossary_examples.keys():
#                         glossary_examples[key].append(SchemaFieldUrn(table_urn, field["fieldPath"]))
#                     else:
#                         glossary_examples[key] = [SchemaFieldUrn(table_urn, field["fieldPath"])]
#             else:
#                 continue
# terms_without_examples = set(glossary_query_results.keys()) - set(glossary_examples.keys())

In [160]:
len(terms_without_examples)

47

In [ ]:
for term_urn, value in glossary_info.glossary.items():
    if term_urn not in terms_without_examples:
        column_urn = glossary_examples[term_urn][0]
        glossary_info.glossary[term_urn]["example"] = glossary_examples[term_urn]

In [ ]:
with open("care_glossary_info_with_examples_and_user_and_member_id_descriptions.pkl", "wb") as f:
    pickle.dump(glossary_info, f)

In [234]:
# glossary_examples['urn:li:glossaryTerm:allergies'][0].urn()

In [233]:
# graph_client.get_entity_semityped(glossary_examples['urn:li:glossaryTerm:allergies'][0].urn())

In [230]:
# from datahub.metadata.schema_classes import DatasetFieldProfileClass, SchemaFieldInfoClass, SchemaFieldClass
# graph_client.get_aspect(entity_urn=glossary_examples['urn:li:glossaryTerm:allergies'][0].urn(), aspect_type=SchemaFieldClass)

In [231]:
# graph_client.get_entity_semityped('urn:li:dataset:(urn:li:dataPlatform:mysql,backup_care.MEMBER_SEGMENTATION_CHILDREN_DETAILS,PROD)')
# # glossary_examples['urn:li:glossaryTerm:allergies'][0].urn()

In [232]:
# for term in terms_without_examples:
#     print(glossary_query_results[term])

In [ ]:
# no assignment- negative
# assignment - positive